CAPA DE BRONZE

En esta fase se cargan los datos tal cual y se mira que coindican nombres de variables, tipos de variables.
Se genera una sola tabla con las bases de datos de ambos tipos de taxis. Para ello se generan columnas de trazabilidad para saber siempre de donde proviene cada registro.

In [14]:
!pip install -q pyspark
import os, re, glob
from functools import reduce
from pyspark.sql import SparkSession, functions as F
spark = (SparkSession.builder
         .master("local[*]")
         .appName("bronze")
         .config("spark.sql.sources.partitionOverwriteMode", "dynamic")
         .getOrCreate())

In [2]:
# Conectar a la carpeta del drive
from google.colab import drive

drive.mount('/content/drive')
BASE = "/content/drive/MyDrive/nyc-taxi"

Mounted at /content/drive


In [31]:
# Rutas archivos
yellow = spark.read.parquet(f"{BASE}/yellow_tripdata_2025-01.parquet")
green  = spark.read.parquet(f"{BASE}/green_tripdata_2025-01.parquet")

yellow.printSchema()
green.printSchema()
print(yellow.count(), green.count())

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)

root
 |-- VendorID: integer (nullable = true)
 |-- lpep_pickup_datetime: ti

La gran mayoría de las columnas aparecen en ambas bases de datos, y los tipos de columnas coinciden en estas.

1.   Hay que reenombrar las columans qu ehacen referencia a la fecha y hora de recogida y llegada
2.   Hay que tener todos los nombres de las variables en minusculas.
3.   Hay que añadir las columnas de trazabilidad.







In [12]:
# poner en minusculas todos los nombres de variables

yellow = yellow.toDF(*[c.lower() for c in yellow.columns])
green = green.toDF(*[c.lower() for c in green.columns])
print(yellow.schema)
print(green.schema)

StructType([StructField('vendorid', IntegerType(), True), StructField('tpep_pickup_datetime', TimestampNTZType(), True), StructField('tpep_dropoff_datetime', TimestampNTZType(), True), StructField('passenger_count', LongType(), True), StructField('trip_distance', DoubleType(), True), StructField('ratecodeid', LongType(), True), StructField('store_and_fwd_flag', StringType(), True), StructField('pulocationid', IntegerType(), True), StructField('dolocationid', IntegerType(), True), StructField('payment_type', LongType(), True), StructField('fare_amount', DoubleType(), True), StructField('extra', DoubleType(), True), StructField('mta_tax', DoubleType(), True), StructField('tip_amount', DoubleType(), True), StructField('tolls_amount', DoubleType(), True), StructField('improvement_surcharge', DoubleType(), True), StructField('total_amount', DoubleType(), True), StructField('congestion_surcharge', DoubleType(), True), StructField('airport_fee', DoubleType(), True), StructField('cbd_congestio

In [13]:
# renombrar las columnas de fecha

yellow = yellow.withColumnsRenamed({"tpep_pickup_datetime": "pickup_datetime", "tpep_dropoff_datetime": "dropoff_datetime"})

green = green.withColumnsRenamed({"lpep_pickup_datetime": "pickup_datetime", "lpep_dropoff_datetime": "dropoff_datetime"})

print(yellow.schema)
print(green.schema)

StructType([StructField('vendorid', IntegerType(), True), StructField('pickup_datetime', TimestampNTZType(), True), StructField('dropoff_datetime', TimestampNTZType(), True), StructField('passenger_count', LongType(), True), StructField('trip_distance', DoubleType(), True), StructField('ratecodeid', LongType(), True), StructField('store_and_fwd_flag', StringType(), True), StructField('pulocationid', IntegerType(), True), StructField('dolocationid', IntegerType(), True), StructField('payment_type', LongType(), True), StructField('fare_amount', DoubleType(), True), StructField('extra', DoubleType(), True), StructField('mta_tax', DoubleType(), True), StructField('tip_amount', DoubleType(), True), StructField('tolls_amount', DoubleType(), True), StructField('improvement_surcharge', DoubleType(), True), StructField('total_amount', DoubleType(), True), StructField('congestion_surcharge', DoubleType(), True), StructField('airport_fee', DoubleType(), True), StructField('cbd_congestion_fee', Do

In [20]:
# añadir columnas de trazabilidad
yellow = (yellow.withColumn("taxi_type", F.lit("yellow"))
                .withColumn("source_file", F.lit("yellow_tripdata_2025-01.parquet"))
                .withColumn("ingested_at", F.current_timestamp())
                .withColumn("year_month", F.lit("2025-01"))
)
green = (green.withColumn("taxi_type", F.lit("green"))
                .withColumn("source_file", F.lit("green_tripdata_2025-01.parquet"))
                .withColumn("ingested_at", F.current_timestamp())
                .withColumn("year_month", F.lit("2025-01"))
)
print(yellow.columns)
print(green.columns)

['vendorid', 'pickup_datetime', 'dropoff_datetime', 'passenger_count', 'trip_distance', 'ratecodeid', 'store_and_fwd_flag', 'pulocationid', 'dolocationid', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge', 'airport_fee', 'cbd_congestion_fee', 'taxi_type', 'source_file', 'ingested_at', 'year_month']
['vendorid', 'pickup_datetime', 'dropoff_datetime', 'store_and_fwd_flag', 'ratecodeid', 'pulocationid', 'dolocationid', 'passenger_count', 'trip_distance', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'ehail_fee', 'improvement_surcharge', 'total_amount', 'payment_type', 'trip_type', 'congestion_surcharge', 'cbd_congestion_fee', 'taxi_type', 'source_file', 'ingested_at', 'year_month']


In [23]:
# unir los datos de ambos tipos de taxi

bronze = yellow.unionByName(green, allowMissingColumns=True)

bronze.printSchema()
print(bronze.count())

root
 |-- vendorid: integer (nullable = true)
 |-- pickup_datetime: timestamp_ntz (nullable = true)
 |-- dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- ratecodeid: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- pulocationid: integer (nullable = true)
 |-- dolocationid: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)
 |-- taxi_type: string (nullable = false)
 |-- source_file: string (nullable = false)


In [26]:
# guardar los datos

(bronze.write
      .mode("overwrite")
      .partitionBy("taxi_type", "year_month")
      .parquet(f"{BASE}/bronze")
      )


In [30]:
# comprobación de datos
df =bronze.groupBy("taxi_type").count()
df.show()

+---------+-------+
|taxi_type|  count|
+---------+-------+
|   yellow|3475226|
|    grenn|  48326|
+---------+-------+



Ya se han realizados las modificaciones necesarias en esta fase.

Se ha guardado el archvio diferenciando tipología de taxi y fecha temporal de los datos segun TLC.

Para la capa silver solo hay quue leer carpeta bronze y Spark ya cargara el resto en una sola tabla

Tempos de esta ejecución (solo un mes pro tipología de taxi)

*   Cargar ficheros: 2s
*   Guardar ficheros con todas las modificaciones: 35s
*   El resto ejecuciones, inmediatas




